In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Import packages and modules
import numpy as np
import torch
from copy import deepcopy
from RL4CRN_Feedback.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN_Feedback.Policies.BimolecularMassActionPolicy import BimolecularMassActionPolicy
from RL4CRN_Feedback.Agents.BimolecularMassActionAgent import BimolecularMassActionAgent
from RL4CRN_Feedback.Environments.CRNEnvironment import CRNEnvironment
from RL4CRN_Feedback.Environments.VecCRNEnvironment import VecCRNEnvironment
from RL4CRN_Feedback.Environments.VecCRNEnvironment import SerialVecCRNEnvironment

In [3]:
# Construct the basic CRN
species_labels = ['X_1', 'X_2', 'X_3']
inputs_labels = ['u_1', 'u_2', 'u_3', 'u_4']
stoichiometry_reactants = np.array([[0], [1], [0]], dtype=np.int8)
stoichiometry_products = np.array([[1], [1], [0]], dtype=np.int8)
k = 1
parameters = np.array([k], dtype=np.float32)
input_influence_matrix = np.array([[0], [0], [0], [0]], dtype=np.int8)
outputs = np.array([1], dtype=np.int8)
IOCRN_template = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)
IOCRN_template.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 0, 'product2 index': 0, 'input influence index': 0, 'rate constant':0.1}, mode='species index')
IOCRN_template.add_reaction({'reactant1 index': 1, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.5}, mode='species index')
IOCRN_template.add_reaction({'reactant1 index': 1, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 2, 'rate constant':0.7}, mode='species index')
print('IOCRN template:')
IOCRN_template.print_reactions()

In [4]:
# Construct parallel environments
IOCRN_0 = deepcopy(IOCRN_template)
max_num_reactions = 3
batch_size = 5
N_CPUs = 128      
logger = None                                    
vec_env = VecCRNEnvironment([CRNEnvironment(IOCRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(batch_size)], N_CPUs=N_CPUs, logger=logger)
# vec_env = SerialVecCRNEnvironment([CRNEnvironment(IOCRN_0, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(batch_size)], logger=logger)

In [5]:
# Construct the Agent
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_species = 3; num_inputs = 4
encoder_attributes = {"hidden_size": 64, "num_layers": 2}
structure_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
rate_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
input_influence_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
hidden_size = 1024
num_possible_reactions = IOCRN_0.get_reactions_range()
PolicyModel = BimolecularMassActionPolicy(num_possible_reactions, num_inputs, encoder_attributes, hidden_size, structure_decoder_attributes, rate_decoder_attributes, input_influence_decoder_attributes, continuous_distribution='lognormal', allow_input_influence=True)
Agent = BimolecularMassActionAgent(vec_env.envs[0], PolicyModel, allow_input_influence=False, logger=logger, learning_rate=1e-5, entropy_weight=0.01, entropy_update_coefficient=0.9, entropy_schedule=20, minimum_entropy_weight=1, risk=0.8, risk_update=0.00, max_risk=1.00, risk_schedule=20, device=device)

In [6]:
# Collect observations from the batch of CRNs
observation_batch = vec_env.observe()
reactions_indices_batch, parameters_batch, reactions_indices_influenced_by_inputs_batch = observation_batch

for i in range(batch_size):
    print(f"Reactions Indices for IOCRN {i}: {reactions_indices_batch[i]}")
    print(f"Parameters for IOCRN {i}: {parameters_batch[i]}")
    for j in range(num_inputs):
        print(f"Reactions Indices influenced by input {j+1} for IOCRN {i}: {reactions_indices_influenced_by_inputs_batch[j][i]}")
    print("---------------------")

In [ ]:
# Generate actions using the agent
actions = Agent.act(observation_batch)

# Visualize the actions
for i in range(batch_size):
    print(f"Actions for IOCRN {i}:")
    print(f"Reaction Indices: {actions[i]['reaction index']} \t Reaction: X_{IOCRN_0.map_reaction_to_species(actions[i]['reaction index'][0])} + X_{IOCRN_0.map_reaction_to_species(actions[i]['reaction index'][1])} -> X_{IOCRN_0.map_reaction_to_species(actions[i]['reaction index'][2])} + X_{IOCRN_0.map_reaction_to_species(actions[i]['reaction index'][3])}")
    print(f"Reaction Rates: {actions[i]['rate constant']}")
    print(f"Input Influence Indices: {actions[i]['input influence index']}")
    print("---------------------")

In [9]:
# Take a step in the environment
vec_env.reset()
out = vec_env.step(actions, mode='reaction index')

# Print the results
for i in range(batch_size):
    print(f"Results for IOCRN {i}:")
    vec_env.envs[i].state.print_reactions()
    print("---------------------")